### Fine-Tuning a model the PyTorch Way ###

In [5]:
# HF access authorization

from hf_auth import init_hf

# Initialize the token securely before loading any models
init_hf()

✅ Hugging Face token successfully loaded!


True

In [6]:
from transformers import AutoModelForSequenceClassification, get_scheduler
from torch.optim import AdamW 
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification

import warnings
warnings.filterwarnings('ignore')

In [7]:
# Load and preprocess data

raw_datasets = load_dataset("nyu-mll/glue", "mrpc")

checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [8]:
tokenized_datasets['train']

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 3668
})

#### Prepare for training ####

Before actually writing our training loop, we will need to define a few objects. The first ones are the dataloaders we will use to iterate over batches. But before we can define those dataloaders, we need to apply a bit of postprocessing to our tokenized_datasets, to take care of some things that the Trainer did for us automatically. Specifically, we need to:

    Remove the columns corresponding to values the model does not expect (like the sentence1 and sentence2 columns).
    Rename the column label to labels (because the model expects the argument to be named labels).
    Set the format of the datasets so they return PyTorch tensors instead of lists.

Our tokenized_datasets has one method for each of those steps:

In [9]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names # Check final column names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

#### Define our dataloaders:

In [10]:
train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=8, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=8, collate_fn=data_collator
)

To quickly check there is no mistake in the data processing, we can inspect a batch like this:

In [11]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([8]),
 'input_ids': torch.Size([8, 81]),
 'token_type_ids': torch.Size([8, 81]),
 'attention_mask': torch.Size([8, 81])}

We instantiate the model exactly as we have done before.

In [12]:
# Instantiate model

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
optimizer = AdamW(model.parameters(), lr=3e-5) # AdamW is Adam but with an additional weight decay element built in.

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

num_epochs = 3
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

progress_bar = tqdm(range(num_training_steps))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  0%|          | 0/1377 [00:00<?, ?it/s]

#### Train model ####

In [14]:
model.train()
for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

The import evaluate statement belongs to the Hugging Face evaluate library.<br>
This library provides a standardized way to compute and report various metrics across different machine learning tasks, making it easier to evaluate your models.
Key features of the Hugging Face evaluate library:
 - Wide Range of Metrics: It supports a broad collection of metrics for tasks like text classification, question answering, summarization, translation, and more.
 -  Easy Integration: It seamlessly integrates with popular machine learning frameworks like PyTorch, TensorFlow, and JAX.
 - Load Metrics by Name: You can load metrics directly by their names (e.g., "accuracy", "f1", "bleu"), simplifying the evaluation process.
 Computation and Aggregation:
 - It handles both the computation of metrics on individual examples and the aggregation of results across your dataset.

#### Evaluate Model

In [15]:
import evaluate

metric = evaluate.load("glue", "mrpc")
model.eval()
for batch in eval_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    metric.add_batch(predictions=predictions, references=batch["labels"])

metric.compute()

{'accuracy': 0.8406862745098039, 'f1': 0.8900169204737732}

In [19]:
# Save the model
import os

# path = './pytorchnlp'

# # Create the directory if it doesn't exist
# os.makedirs(path, exist_ok=True)

# # Save your model (state_dict or entire model)
# model_save_path = os.path.join(path, 'model.pth')

# torch.save(model, model_save_path)
path = './pytorchnlp'
os.makedirs(path, exist_ok=True)
model_save_path = os.path.join(path, 'model.pth')

# 1. Save ONLY the weights (state_dict)
torch.save(model.state_dict(), model_save_path)

#### Get predictions

In [20]:
sentence1 = "The character acted very well."
sentence2 = "Characters acting was good."

next_pair_1 = 'The movie was quite bad'

# Tokenize both sentences
inputs = tokenizer(sentence1, sentence2, return_tensors="pt", padding=True, truncation=True)
inputs = inputs.to(device)

In [21]:
# model = torch.load('./pytorchnlp/model.pth')
# model.eval()  # Set the model to evaluation mode
# with torch.no_grad():
#     logits = model(**inputs).logits
# print(logits)

# 2. To load: Re-initialize the blank model architecture first
from transformers import BertForSequenceClassification


# (Ensure your_config matches the exact configuration of your original model)
model = BertForSequenceClassification.from_pretrained('bert-base-uncased') 

# 3. Load the weights into it (This works perfectly with the default weights_only=True)
model.load_state_dict(torch.load('./pytorchnlp/model.pth'))

model.eval() 
with torch.no_grad():
    logits = model(**inputs).logits
print(logits)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tensor([[-1.9790,  1.6106]])


In [22]:
import torch.nn.functional as F

probabilities = F.softmax(logits, dim=1)
predicted_label = torch.argmax(probabilities).item()
print(predicted_label)

if predicted_label == 1:
    print("The sentences are paraphrases.")
else:
    print("The sentences are not paraphrases.")

1
The sentences are paraphrases.
